# Part 3 : Snowflake CoWork

## このノートブックでやること
1. **データ確認** — SNS 投稿と売上データを確認する
2. **投稿画像の AI 分析** — 画像から特徴量を自動抽出する
3. **Semantic View の作成** — データの意味を定義する
4. **Snowflake CoWork で質問してみよう** — 成功例を体験する
5. **CoWork が答えられない質問をしてみよう** — 失敗例から原因を理解する
6. **Semantic View を改善する** — バズスコアのメトリクスを追加する
7. **再チャレンジ！** — 改善後に同じ質問をして違いを確認する

In [ ]:
-- 環境設定
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE GLACIERSTYLE_WH;
USE DATABASE GLACIERSTYLE_DB;
USE SCHEMA EC_ANALYTICS_SCHEMA;


## 1. データ確認

今回 SI で分析に使うデータを確認しておきましょう。

### 使用するテーブル

| テーブル | 内容 |
|---|---|
| `raw_sns_mentions` | インフルエンサーの SNS 投稿（いいね・RT 数など） |
| `fact_orders` | 注文データ（売上金額・日時など） |
| `dim_products` | 商品マスタ（商品名・カテゴリなど） |

In [ ]:
-- SNS 投稿データを確認
SELECT
    post_id,
    platform,
    username,
    display_name,
    posted_at,
    likes,
    retweets,
    replies,
    content
FROM raw_sns_mentions
LIMIT 5;


In [ ]:
-- 売上データを確認
SELECT
    order_id,
    order_datetime,
    product_id,
    quantity,
    total_amount
FROM fact_orders
LIMIT 5;


## 2. 投稿画像の AI 分析

SNS の投稿には**画像**が添付されています。テキストデータだけでなく、画像の特徴もバズスコアに影響するかもしれません。

Snowflake の AI 関数を使って、投稿画像から特徴を自動抽出します。

### 抽出する特徴

| 特徴 | 内容 |
|---|---|
| 人物の有無 | 人が写っているか |
| 表情 | ポジティブ / ネガティブ / ニュートラル |
| 画像の明るさ | 明るい / 暗い |
| 商品の視認性 | 商品がはっきり映っているか |

In [ ]:
-- ステージ上の投稿画像ファイルを確認
SELECT
    relative_path,
    size,
    last_modified
FROM DIRECTORY(@DATA_STAGE)
WHERE relative_path LIKE 'data/ad_images/%'
ORDER BY relative_path;


In [ ]:
-- 1枚の画像を AI で分析してみる（動作確認）
-- TODO: 画像ファイルパスを実際のものに差し替える
SELECT
    relative_path,
    AI_COMPLETE(
        'claude-3-5-sonnet',
        [
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'image',
                        'source': TO_FILE('@DATA_STAGE', relative_path)
                    },
                    {
                        'type': 'text',
                        'text': '以下の項目を日本語で JSON 形式で回答してください。
- has_person: 人物が写っているか (true/false)
- facial_expression: 表情 (ポジティブ / ネガティブ / ニュートラル / 不明)
- brightness: 画像の明るさ (明るい / 普通 / 暗い)
- product_visibility: 商品の視認性 (高い / 普通 / 低い / 不明)'
                    }
                ]
            }
        ]
    ) AS image_analysis
FROM DIRECTORY(@DATA_STAGE)
WHERE relative_path LIKE 'data/ad_images/%'
LIMIT 1;


In [ ]:
-- 全画像を分析して gold テーブルに保存
-- TODO: image_id と raw_sns_mentions の結合キー設計に合わせて修正する
CREATE OR REPLACE TABLE gold_influencer_image_analysis AS
SELECT
    relative_path                                               AS image_path,
    SPLIT_PART(relative_path, '/', -1)                        AS image_file,
    AI_COMPLETE(
        'claude-3-5-sonnet',
        [
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'image',
                        'source': TO_FILE('@DATA_STAGE', relative_path)
                    },
                    {
                        'type': 'text',
                        'text': '以下の項目を日本語で JSON 形式で回答してください。
- has_person: 人物が写っているか (true/false)
- facial_expression: 表情 (ポジティブ / ネガティブ / ニュートラル / 不明)
- brightness: 画像の明るさ (明るい / 普通 / 暗い)
- product_visibility: 商品の視認性 (高い / 普通 / 低い / 不明)'
                    }
                ]
            }
        ]
    )                                                          AS analysis_raw,
    analysis_raw['has_person']::BOOLEAN                       AS has_person,
    analysis_raw['facial_expression']::VARCHAR                AS facial_expression,
    analysis_raw['brightness']::VARCHAR                       AS brightness,
    analysis_raw['product_visibility']::VARCHAR               AS product_visibility
FROM DIRECTORY(@DATA_STAGE)
WHERE relative_path LIKE 'data/ad_images/%';

-- 結果確認
SELECT * FROM gold_influencer_image_analysis LIMIT 5;


## 3. Semantic View を作成する

**Semantic View（セマンティックビュー）** は、CoWork が「このデータは何を意味するか」を理解するためのメタデータ定義です。

- テーブルの意味や関係性を定義する
- ビジネス指標（メトリクス）をあらかじめ定義しておく
- CoWork はこの定義をもとに自然言語クエリを SQL に変換する

### ポイント

Semantic View に **定義されていないメトリクスは CoWork が答えられません。**
後のセクションでこれを体験します。

### 今回作成する Semantic View

| 要素 | 内容 |
|---|---|
| ベーステーブル | `raw_sns_mentions`（SNS 投稿） |
| 関連テーブル | `fact_orders`（売上）, `dim_products`（商品）, `gold_influencer_image_analysis`（画像分析） |
| ディメンション | 投稿プラットフォーム、ユーザー名、商品カテゴリ、画像特徴 |
| メトリクス | 合計売上、注文件数、投稿件数 |

In [ ]:
-- ========================================================
-- Semantic View の作成（ドラフト / TODO: 内容差し替え予定）
-- ========================================================
CREATE OR REPLACE SEMANTIC VIEW sv_influencer_analysis
  TABLES (
    raw_sns_mentions PRIMARY (
      DIMENSIONS (
        post_id          COMMENT '投稿ID',
        platform         COMMENT '投稿プラットフォーム（Instagram / X など）',
        username         COMMENT 'インフルエンサーのユーザー名',
        display_name     COMMENT '表示名',
        posted_at        COMMENT '投稿日時',
        hashtags         COMMENT 'ハッシュタグ'
      )
      METRICS (
        total_posts    AS COUNT(post_id)       COMMENT '総投稿件数',
        total_likes    AS SUM(likes)           COMMENT '合計いいね数',
        total_retweets AS SUM(retweets)        COMMENT '合計リツイート数',
        total_replies  AS SUM(replies)         COMMENT '合計リプライ数'
      )
    ),
    fact_orders (
      DIMENSIONS (
        order_id        COMMENT '注文ID',
        order_datetime  COMMENT '注文日時',
        product_id      COMMENT '商品ID',
        order_channel   COMMENT '注文チャネル'
      )
      METRICS (
        total_revenue   AS SUM(total_amount)   COMMENT '合計売上金額',
        total_orders    AS COUNT(order_id)     COMMENT '注文件数'
      )
    ),
    dim_products (
      DIMENSIONS (
        product_id    COMMENT '商品ID',
        product_name  COMMENT '商品名',
        category_l1   COMMENT '商品カテゴリ（大）',
        category_l2   COMMENT '商品カテゴリ（中）',
        brand         COMMENT 'ブランド名'
      )
    )
  )
  RELATIONSHIPS (
    fact_orders (product_id) REFERENCES dim_products (product_id)
    -- TODO: SNS 投稿と注文の関連付け方法を検討する
  );


In [ ]:
-- 作成した Semantic View を確認
DESCRIBE SEMANTIC VIEW sv_influencer_analysis;


## 4. Snowflake CoWork で質問してみよう（成功例）

Semantic View の準備ができました。いよいよ CoWork を使ってみましょう。

### CoWork の起動方法

1. Snowsight の左メニューから **「Snowflake CoWork」** をクリック
2. 先ほど作成した `sv_influencer_analysis` を選択
3. チャット欄に質問を入力

<!-- TODO: スクリーンショット画像を追加 -->
<!-- ![CoWork Launch](images/part3/01_cowork_launch.png) -->

### 試してみる質問（シンプルな集計）

まずは CoWork が確実に答えられる**シンプルな質問**を試してみましょう。

```
売上の高い商品カテゴリ上位5つを教えてください
```

```
先月の注文件数は何件ですか？
```

```
投稿数が最も多いインフルエンサーは誰ですか？
```

> **確認ポイント:** CoWork が SQL を自動生成して結果を返してくれることを確認しましょう。

In [ ]:
-- 参考: CoWork が生成する SQL と同等のクエリを手動で確認
-- CoWork の回答と結果が一致するか検証に使えます
SELECT
    p.category_l1,
    SUM(o.total_amount) AS total_revenue
FROM fact_orders o
JOIN dim_products p ON o.product_id = p.product_id
GROUP BY p.category_l1
ORDER BY total_revenue DESC
LIMIT 5;


## 5. CoWork が答えられない質問をしてみよう（失敗例）

今度は少し**踏み込んだ質問**を CoWork にしてみましょう。

### 試してみる質問

```
インフルエンサーのバズスコアと売上の相関を見せてください
```

```
バズった投稿の翌日の売上はどれくらい増えましたか？
```

```
明るい画像の投稿はバズりやすいですか？
```

> **予想:** CoWork はこれらの質問に答えられないか、的外れな回答をするはずです。

### なぜ答えられないのか？

**「バズスコア」という概念が Semantic View に定義されていないから**です。

- CoWork は Semantic View の定義をもとに SQL を組み立てます
- `buzz_score`（バズスコア）というメトリクスが存在しない
- 画像分析結果と SNS 投稿の関連付けも定義されていない

→ **Semantic View を改善して、このような質問にも答えられるようにしましょう！**

## 6. Semantic View を改善する

**バズスコア** のメトリクスを Semantic View に追加します。

### バズスコアの定義

バズスコアは以下のように定義します：

```
buzz_score = likes + retweets * 2 + replies
```

※ リツイートはリーチへの影響が大きいため 2 倍の重みを設定

### 改善するポイント

1. `buzz_score` メトリクスの追加
2. `gold_influencer_image_analysis`（画像分析結果）テーブルの追加
3. Verified Query（検証済みクエリ）の追加
   - CoWork が「バズスコア」「明るい画像」と聞かれたとき、正しい SQL を使うためのヒント集

> **Verified Query とは？**
> よく聞かれる質問とその正解 SQL をペアで登録することで、CoWork の回答精度が向上します。

In [ ]:
-- ========================================================
-- Semantic View を改善版に再作成（buzz_score メトリクスを追加）
-- TODO: 内容差し替え予定
-- ========================================================
CREATE OR REPLACE SEMANTIC VIEW sv_influencer_analysis
  TABLES (
    raw_sns_mentions PRIMARY (
      DIMENSIONS (
        post_id          COMMENT '投稿ID',
        platform         COMMENT '投稿プラットフォーム（Instagram / X など）',
        username         COMMENT 'インフルエンサーのユーザー名',
        display_name     COMMENT '表示名',
        posted_at        COMMENT '投稿日時',
        hashtags         COMMENT 'ハッシュタグ'
      )
      METRICS (
        total_posts    AS COUNT(post_id)                          COMMENT '総投稿件数',
        total_likes    AS SUM(likes)                             COMMENT '合計いいね数',
        total_retweets AS SUM(retweets)                          COMMENT '合計リツイート数',
        total_replies  AS SUM(replies)                           COMMENT '合計リプライ数',
        buzz_score     AS SUM(likes + retweets * 2 + replies)    COMMENT 'バズスコア（いいね + RT×2 + リプライ）'
      )
    ),
    fact_orders (
      DIMENSIONS (
        order_id        COMMENT '注文ID',
        order_datetime  COMMENT '注文日時',
        product_id      COMMENT '商品ID',
        order_channel   COMMENT '注文チャネル'
      )
      METRICS (
        total_revenue   AS SUM(total_amount)   COMMENT '合計売上金額',
        total_orders    AS COUNT(order_id)     COMMENT '注文件数'
      )
    ),
    dim_products (
      DIMENSIONS (
        product_id    COMMENT '商品ID',
        product_name  COMMENT '商品名',
        category_l1   COMMENT '商品カテゴリ（大）',
        category_l2   COMMENT '商品カテゴリ（中）',
        brand         COMMENT 'ブランド名'
      )
    )
  )
  RELATIONSHIPS (
    fact_orders (product_id) REFERENCES dim_products (product_id)
  )
  VERIFIED QUERIES (
    -- TODO: Verified Query を追加する
    -- 例:
    --   question: 'インフルエンサーのバズスコアランキングを教えてください'
    --   sql: SELECT username, SUM(likes + retweets * 2 + replies) AS buzz_score
    --        FROM raw_sns_mentions GROUP BY username ORDER BY buzz_score DESC
  );


In [ ]:
-- バズスコアの計算結果を手動で確認（期待値チェック）
SELECT
    username,
    display_name,
    SUM(likes)                          AS total_likes,
    SUM(retweets)                       AS total_retweets,
    SUM(replies)                        AS total_replies,
    SUM(likes + retweets * 2 + replies) AS buzz_score
FROM raw_sns_mentions
GROUP BY username, display_name
ORDER BY buzz_score DESC
LIMIT 10;


## 7. 再チャレンジ！（改善後）

Semantic View を改善しました。もう一度 CoWork に同じ質問をしてみましょう。

### 再度試してみる質問

```
インフルエンサーのバズスコアランキングを見せてください
```

```
明るい画像の投稿はバズりやすいですか？
```

```
カテゴリ別のバズスコアと売上の関係を教えてください
```

> **確認ポイント:**
> - 先ほど答えられなかった質問に、今度は正しく回答できるか確認しましょう
> - CoWork が自動生成した SQL を見て、期待通りの集計になっているか確認しましょう

### まとめ

| | 改善前 | 改善後 |
|---|---|---|
| バズスコアの質問 | ❌ 答えられない | ✅ 正しく回答 |
| 画像特徴との相関 | ❌ 答えられない | ✅ 正しく回答 |
| シンプルな集計 | ✅ 正しく回答 | ✅ 正しく回答 |

**Semantic View の定義がそのまま CoWork の「答えられる範囲」になります。**
ビジネス要件が追加されるたびに Semantic View を育てていくことで、CoWork はどんどん賢くなります。

## お疲れ様でした！

### Part 3 で学んだこと

- **Snowflake CoWork** を使って自然言語でデータに質問できた
- **Semantic View** の定義が CoWork の回答精度に直結することを体験した
- メトリクスの追加（バズスコア）によって CoWork の回答範囲を広げた

### 次のステップ

- Part 4: Snowflake Marketplace（外部データとの連携）